# Jacobians, Elasticities and Comparative Statics

The Jacobian of a system of equations collects every marginal effect. Scaled by $x/F$ it becomes a matrix of elasticities. Inverted at an equilibrium, it gives comparative statics through the implicit function theorem.

In [ ]:
Pollis.packages("DifferentiationInterface", "ForwardDiff")

In [ ]:
using DifferentiationInterface, LinearAlgebra
import ForwardDiff
backend = AutoForwardDiff()

## A two-good demand system

Demand for goods 1 and 2 depends on both prices $p$ and income $m$: $x = D(p_1, p_2, m)$.

In [ ]:
D(z) = [z[3] * 0.4 / z[1] * (z[2] / z[1])^0.2,
        z[3] * 0.6 / z[2] * (z[1] / z[2])^0.1]
z0 = [2.0, 3.0, 100.0]   # p1, p2, m
J = jacobian(D, backend, z0)

## Elasticities

$E_{ij} = \dfrac{\partial D_i}{\partial z_j}\dfrac{z_j}{D_i}$: own-price elasticities on the diagonal of the price block, cross-price elasticities off it, income elasticities in the last column.

In [ ]:
E = J .* z0' ./ D(z0)
(own_price = (E[1, 1], E[2, 2]), cross_price = (E[1, 2], E[2, 1]), income = E[:, 3], homogeneity_check = sum(E, dims=2))

Demand is homogeneous of degree zero in prices and income, so each row of elasticities sums to zero (Euler's theorem).

## Comparative statics of a market equilibrium

Supply $S(p) = 2p$, demand $D(p, t) = 20 - p - t$ with a tax-like shifter $t$. The equilibrium solves $G(p, t) = S(p) - D(p, t) = 0$. By the implicit function theorem $\dfrac{dp}{dt} = -\dfrac{\partial G/\partial t}{\partial G/\partial p}$.

In [ ]:
G(v) = [2v[1] - (20 - v[1] - v[2])]   # v = [p, t]
t0 = 1.0; p_star = (20 - t0) / 3
Jg = jacobian(G, backend, [p_star, t0])
dp_dt = -Jg[1, 2] / Jg[1, 1]
(equilibrium_price = p_star, dp_dt = dp_dt, exact = -1 / 3)

## Exercises

1. Make supply nonlinear, $S(p) = p^{1.5}$, solve with the Nonlinear Systems panel and compute $dp/dt$ the same way.
2. Compute the elasticities of a Cobb-Douglas utility's demand functions and check they are exactly $-1$ and $1$.